# Zip files adjusts

In [ ]:
#Download IGS_TEC_maps_intersection_raw_files_case_study_Dec_2024.zip from https://doi.org/10.5281/zenodo.15453941

In [ ]:
%pwd

In [ ]:
!ls

In [ ]:
!unzip 'IGS_TEC_maps_intersection_raw_files_case_study_Dec_2024.zip'

In [ ]:
%cd 'IGS_TEC_maps_intersection_raw_files_case_study_Dec_2024'

# Data files adjusts

In [ ]:
import os
import numpy as np

diretorio_atual = './'

arquivos_24i = []

for root, dirs, files in os.walk(diretorio_atual):
    for file in files:
        if file.endswith(".24i") and file.startswith("igsg"):
            arquivos_24i.append(os.path.join(root, file))

arquivos_24i = sorted(arquivos_24i)
print(arquivos_24i)

print("Arquivos .24i encontrados (ordenados):")
for arquivo in arquivos_24i:
    print(arquivo)

for caminho_arquivo_i in arquivos_24i:

    lista_dados = []

    with open(caminho_arquivo_i, 'r') as arquivo:
        for linha in arquivo:
            if "START OF RMS MAP" in linha:
                print("Fim do arquivo TEC encontrado. Interrompendo o processamento.")
                break
            if "LAT/LON1/LON2/DLON/H" in linha:

                valores_linha1 = list(map(float, arquivo.readline().split()))
                valores_linha2 = list(map(float, arquivo.readline().split()))
                valores_linha3 = list(map(float, arquivo.readline().split()))
                valores_linha4 = list(map(float, arquivo.readline().split()))
                valores_linha5 = list(map(float, arquivo.readline().split()))

                valores_combinados = valores_linha1 + valores_linha2 + valores_linha3 + valores_linha4 + valores_linha5

                lista_dados.append(valores_combinados)

        array_dados = np.array(lista_dados)

        nome_arquivo_sem_extensao = os.path.splitext(os.path.basename(caminho_arquivo_i))[0]

        caminho_saida_npy = os.path.join(os.path.dirname(caminho_arquivo_i), f"{nome_arquivo_sem_extensao}_24.npy")

        elementos_por_linha = 71

        total_linhas = array_dados.shape[0] // elementos_por_linha

        new_shape = (total_linhas, elementos_por_linha, array_dados.shape[1])
        array_dados = array_dados[:total_linhas * elementos_por_linha, :].reshape(new_shape)
        print(np.shape(array_dados))
        array_dados = array_dados[:12, 19:67+1, 14:36+1]
        print(np.shape(array_dados))
        np.save(caminho_saida_npy, array_dados)

        print(f"Array salvo em: {caminho_saida_npy}")

In [ ]:
%pwd

In [ ]:
diretorio_principal = './'

arquivos_npy = [arquivo for arquivo in os.listdir(diretorio_principal) if arquivo.endswith('_24.npy')]
arquivos_npy = sorted(arquivos_npy)
print(arquivos_npy)

if not arquivos_npy:
    print("Nenhum arquivo .npy encontrado no diretório atual.")
else:
    array_4d = np.empty((len(arquivos_npy), 12, 49, 23))

    for i, arquivo in enumerate(arquivos_npy):
        caminho_arquivo = os.path.join(diretorio_principal, arquivo)
        array_npy = np.load(caminho_arquivo)
        array_4d[i, :, :, :] = array_npy

    array_4d = array_4d/10
    np.save(os.path.join(diretorio_principal, 'IGS_TEC_maps_intersection_case_study_Dec_2024.npy'), array_4d)

    print(f"Processados {len(arquivos_npy)} arquivos .npy")
    print(f"Forma do array resultante: {array_4d.shape}")

In [ ]:
igs_2024 = np.load('IGS_TEC_maps_intersection_case_study_Dec_2024.npy').astype(np.float64)
print(np.shape(igs_2024))
print(type(igs_2024))
print(type(igs_2024[0]))
print(type(igs_2024[0][0]))
print(type(igs_2024[0][0][0]))
print(type(igs_2024[0][0][0][0]))

# Dates adjusts

In [ ]:
import numpy as np
from datetime import datetime, timedelta

def doy_to_date(year, doy):
    return datetime(year, 1, 1) + timedelta(days=doy - 1)

doys = {

    2024: {
        12: [336, 337, 338, 339, 340, 341, 342, 343, 344, 345,
             346, 347, 349, 350, 351, 352, 353, 354, 355, 356,
             357, 358, 359, 360, 361, 362, 363, 364, 365]
    }
}

datetime_list = []

for year in doys.keys():
    for month in doys[year].keys():
        for doy in doys[year][month]:
            base_date = doy_to_date(year, doy)
            for hour in range(0, 24, 2):
                dt = datetime(base_date.year, base_date.month, base_date.day, hour, 0, 0)
                datetime_list.append(dt)

igs_datetimes_2024 = np.array(datetime_list, dtype='datetime64[s]')

print(f"Total number of datetime points: {len(igs_datetimes_2024)}")
print(f"First datetime: {igs_datetimes_2024[0]}")
print(f"Last datetime: {igs_datetimes_2024[-1]}")

print("\nSample of first 10 datetime entries:")
for dt in igs_datetimes_2024[:10]:
    print(dt)

print("\nSample of last 10 datetime entries:")
for dt in igs_datetimes_2024[-10:]:
    print(dt)

In [ ]:
igs_datetimes_2024.shape

In [ ]:
igs_datetimes_2024

In [ ]:
igs_2024.shape

In [ ]:
print(len(igs_2024.tolist()))
print(len(igs_2024[0].tolist()))
print(len(igs_2024[0][0].tolist()))
print(len(igs_2024[0][0][0].tolist()))

In [ ]:
igs_2024.shape

In [ ]:
igs_2024_shaped = np.reshape(igs_2024, (-1, 49, 23))

In [ ]:
np.shape(igs_2024_shaped)

In [ ]:
np.shape(igs_2024_shaped.tolist())

## Create dataframe

In [ ]:
igs_datetimes_2024.shape

In [ ]:
igs_2024_shaped.shape

In [ ]:
import pandas as pd

data = {
    'DATETIME': igs_datetimes_2024,
    'TECMAP': igs_2024_shaped.tolist()
}

df_mapas_igs_2024 = pd.DataFrame(data)

df_mapas_igs_2024

In [ ]:
np.array(df_mapas_igs_2024.iloc[0]['TECMAP'])

In [ ]:
np.array(df_mapas_igs_2024.iloc[0]['TECMAP']).shape

In [ ]:
df_mapas_igs_2024.to_pickle("./TF_IGS_TEC_maps_case_study_Dec_2024.pkl")

In [ ]:
mapas_igs_2024 = np.array(df_mapas_igs_2024.iloc[:]['TECMAP'])

In [ ]:
np.shape(df_mapas_igs_2024)

In [ ]:
np_mapas_igs_2024 = []
for i in range(len(mapas_igs_2024)):
    np_mapas_igs_2024.append(mapas_igs_2024[i])
np_mapas_igs_2024 = np.array(np_mapas_igs_2024)

In [ ]:
np.shape(np_mapas_igs_2024)

In [ ]:
type(np_mapas_igs_2024)

In [ ]:
np.save('TF_IGS_TEC_maps_case_study_Dec_2024.npy', np_mapas_igs_2024)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!ls -lh TF*

In [ ]:
%cp /content/TF_IGS_TEC_maps_case_study_Dec_2024.pkl /content/drive/MyDrive/TF_IGS_TEC_maps_case_study_Dec_2024.pkl

In [ ]:
%cp /content/TF_IGS_TEC_maps_case_study_Dec_2024.npy /content/drive/MyDrive/TF_IGS_TEC_maps_case_study_Dec_2024.npy